In [ ]:
# Import packages
import requests
import json
import jmespath
import pandas as pd
from pandas import DataFrame
import matplotlib.pyplot as plt
from collections import Counter
from bs4 import BeautifulSoup
import time
import csv
import random
import os, re

In [113]:
exec(open("/Users/zhangxiaobei/Documents/GitHub/datasci530fall2025/Lecture 06/scripts/import_packages.py").read())

In [146]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

In [147]:
options = webdriver.ChromeOptions()

In [148]:
url = "https://www.sciencedirect.com/journal/social-networks/issues"
driver.get(url)

In [6]:

panels = driver.find_elements("xpath", '//li[contains(@class,"accordion-panel")]')

volumes = []
vol_dates = []
links = []
for i, panel in enumerate(panels):
    buttons = driver.find_elements("xpath", '//button[contains(@class,"accordion-panel-title")]')
    button = buttons[i]
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
    time.sleep(0.5)
    expanded = button.get_attribute("aria-expanded")
    if expanded != "true": 
        try:
            button.click()
        except:
            driver.execute_script("arguments[0].click();", button)
        time.sleep(1.5)
    #Exract volume and its issue date 
    issue_divs = panel.find_elements("xpath", './/div[contains(@class, "issue-item")]')
    for div in issue_divs:
        try:
            vol = div.find_element("xpath", './/a[contains(@href, "/journal/social-networks/vol/")]')
            vol_date = div.find_element("xpath", './/h3[contains(@class, "js-issue-status")]')

            volume = vol.text.strip()
            date_text = vol_date.text.strip()
            link = vol.get_attribute("href")
            if volume:
                volumes.append(volume)
                vol_dates.append(date_text)
                links.append(link)
        
        except Exception as e:
            print(f"Error at panel {i}: {e}")
            continue
print(volumes)
print(vol_dates)

['Volume 84', 'Volume 83', 'Volume 82', 'Volume 81', 'Volume 80', 'Volume 79', 'Volume 78', 'Volume 77', 'Volume 76', 'Volume 75', 'Volume 74', 'Volume 73', 'Volume 72', 'Volume 71', 'Volume 70', 'Volume 69', 'Volume 68', 'Volume 67', 'Volume 66', 'Volume 65', 'Volume 64', 'Volume 63', 'Volume 62', 'Volume 61', 'Volume 60', 'Volume 59', 'Volume 58', 'Volume 57', 'Volume 56', 'Volume 55', 'Volume 54', 'Volume 53', 'Volume 52', 'Volume 51', 'Volume 50', 'Volume 49', 'Volume 48', 'Volume 47', 'Volume 46', 'Volume 45', 'Volume 44', 'Volume 43', 'Volume 42', 'Volume 41', 'Volume 40', 'Volume 39', 'Volume 38', 'Volume 37', 'Volume 36', 'Volume 35, Issue 4', 'Volume 35, Issue 3', 'Volume 35, Issue 2', 'Volume 35, Issue 1', 'Volume 34, Issue 4', 'Volume 34, Issue 3', 'Volume 34, Issue 2', 'Volume 34, Issue 1', 'Volume 33, Issue 4', 'Volume 33, Issue 3', 'Volume 33, Issue 2', 'Volume 33, Issue 1', 'Volume 32, Issue 4', 'Volume 32, Issue 3', 'Volume 32, Issue 2', 'Volume 32, Issue 1', 'Volume 31

In [ ]:
# minimal changes: append mode + save progress before visiting + simple retry/backoff

OUT_CSV = "social_networks_articles.csv"
PROG = "progress.json"

# 1) 读取已写入的 URL（更稳健）
done_urls = set()
if os.path.exists(OUT_CSV):
    try:
        with open(OUT_CSV, "r", encoding="utf-8") as fr:
            rdr = csv.DictReader(fr)
            # find the article url column
            url_col = None
            for col in rdr.fieldnames or []:
                if col.strip().lower() == "article_url":
                    url_col = col
                    break
            # read the url and add it to done urls set
            if url_col:
                for row in rdr:
                    u = row.get(url_col)
                    if u:
                        done_urls.add(u)
    except Exception as e:
        print("Warning reading existing CSV for done_urls:", e)

# read progress
start_vol = 0
start_art = 0
if os.path.exists(PROG):
    try:
        with open(PROG, "r", encoding="utf-8") as pf:
            prog = json.load(pf)
            start_vol = int(prog.get("vol_idx", 0))
            start_art = int(prog.get("art_idx", 0))
            print("Resuming from progress:", start_vol, start_art)
    except Exception as e:
        print("Warning reading progress.json:", e)

# open or write csv
write_mode = "a" if os.path.exists(OUT_CSV) else "w"
with open(OUT_CSV, write_mode, newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if write_mode == "w": #if first time write csv, define column names
        writer.writerow(["Volume", "Date", "Article", "Authors", "Article_URL", "Type", "Keywords"])
        f.flush()

    # volume loops
    # read links to each volume
    # extract article title, type, and link
    for i in range(start_vol, len(links)):
        link = links[i]
        print(f"\n=== volume {i}: {volumes[i]} -> {link} ===")
        driver.get(link)
        time.sleep(random.uniform(3, 10))

        articles = driver.find_elements("xpath", '//li[contains(@class, "js-article-list-item")]')
        article_metas = []
        for art in articles:
            try:
                title = art.find_element("xpath", './/span[contains(@class, "js-article-title")]').text.strip()
            except:
                title = ""
            try:
                atype = art.find_element("xpath", './/span[contains(@class, "js-article-subtype")]').text.strip()
            except:
                atype = ""
            try:
                link_art = art.find_element("xpath", './/a[contains(@href, "/science/article/pii")]')
                url = link_art.get_attribute("href")
            except:
                url = ""
            if url:
                article_metas.append({"title": title, "type": atype, "url": url}) 

        #starts from last time's last article or the new volume's first article
        art_start_index = start_art if i == start_vol else 0

        #Iterate through each article in the current volume
        for j in range(art_start_index, len(article_metas)):
            meta = article_metas[j]
            article_url = meta["url"]

            # Check if the article has been read
            if article_url in done_urls:
                print("  skip already done:", article_url)
                # update progress to next article
                prog = {"vol_idx": i, "art_idx": j+1}
                tmp = PROG + ".tmp"
                with open(tmp, "w", encoding="utf-8") as pf:
                    json.dump(prog, pf)
                os.replace(tmp, PROG)
                continue

            # write progress in json file
            prog = {"vol_idx": i, "art_idx": j}
            tmp = PROG + ".tmp"
            with open(tmp, "w", encoding="utf-8") as pf:
                json.dump(prog, pf)
            os.replace(tmp, PROG)

            # if blocked, exit the web scrabing 
            success = False
            try:
                driver.get(article_url)
                time.sleep(random.uniform(3, 8))
                page = driver.page_source.lower()
                #deterrmine if the web block
                if "there was a problem providing the content you requested" in page or "access denied" in page:
                    print(f"   Detected block for {article_url}")
                    raise SystemExit("Blocked")
            
                success = True

            except SystemExit:
                    raise
            except Exception as e:
                    print("   get article error:", e)
                    success = False

            # extract keywords and authors
            keywords = ""
            if success:
                try:
                    keywords_eles = driver.find_elements("xpath", '//div[@class = "keyword"]')
                    kw = [k.text.strip() for k in keywords_eles if k.text.strip()]
                    keywords = ", ".join(kw)
                except Exception as e:
                    keywords = ""
                
                try:
                    author = ", ".join(
                        f"{g.text.strip()} {s.text.strip()}".strip()
                        for g, s in zip(
                            driver.find_elements("xpath", '(//*[@id="author-group" or contains(@class,"author-group")])[1]//span[contains(@class,"given-name")]'),
                            driver.find_elements("xpath", '(//*[@id="author-group" or contains(@class,"author-group")])[1]//span[contains(@class,"surname")]')
                        )
                    )
                except:
                    author = ""

            # Write a line to file and done_urls
            writer.writerow([volumes[i], vol_dates[i], meta["title"], author, article_url, meta["type"], keywords])
            f.flush()
            done_urls.add(article_url)

            # Progress to the next
            prog = {"vol_idx": i, "art_idx": j+1}
            tmp = PROG + ".tmp"
            with open(tmp, "w", encoding="utf-8") as pf:
                json.dump(prog, pf)
            os.replace(tmp, PROG)

        # Reset start_art after the volume is completed
        start_art = 0

print("Finished (or exited due to block).")


Resuming from progress: 19 8

=== volume 19: Volume 65 -> https://www.sciencedirect.com/journal/social-networks/vol/65/suppl/C ===

=== volume 20: Volume 64 -> https://www.sciencedirect.com/journal/social-networks/vol/64/suppl/C ===
   Detected block for https://www.sciencedirect.com/science/article/pii/S0378873320300721


SystemExit: Blocked